In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, HTML

BASE_DIR = Path(".").resolve()
MAPPINGS_DIR = BASE_DIR / "acc_mappings" / "acc_mappings_stx"


In [12]:

rows = []

for fp in sorted(MAPPINGS_DIR.glob("*.json")):
    firm_id = fp.stem
    data = json.loads(fp.read_text(encoding="utf-8"))

    for v in data.get("variables", []):
        if v.get("needs_manual_review", False):
            final_choice = v.get("final_choice", [])
            # make final_choice readable
            final_str = "; ".join([f"{x['sheet_name']}::{x['row_label']}" for x in final_choice]) if final_choice else ""
            rows.append({
                "firm_id": firm_id,
                "variable": v.get("variable", ""),
                "notes": v.get("notes", ""),
                "final_choice": final_str,
                "num_candidates": len(v.get("candidates", [])),
            })

if len(rows) == 0:
    print("No manual reviews needed. All mappings have been reviewed and finalized.")
else:
    df_review = pd.DataFrame(rows).sort_values(["firm_id", "variable"]).reset_index(drop=True)

    print(f"Manual review needed for {len(df_review)} firm-variable mappings across {df_review['firm_id'].nunique() if len(df_review) else 0} firms.")
    display(HTML(f"""
    <div style="max-height: 450px; overflow-y: auto; border: 1px solid #ddd; padding: 6px;">
    {df_review.to_html(index=False)}
    </div>
    """))

Manual review needed for 29 firm-variable mappings across 25 firms.


firm_id,variable,notes,final_choice,num_candidates
AMBUb.CO,CHE,"Balance Sheet contains the exact desired label, but it appears twice with different values in the preview, so the mapping may require confirmation of which duplicate row is intended.",Balance Sheet::Cash and cash equivalents,7
ATLA.CO,TXP,"Selected the only row indicating taxes payable, but the label looks merged/corrupted and may mix a current tax payable line with the following non-current liabilities header.",Balance Sheet::Current tax payable Non-current liabilities,1
BO.CO,STD,Selected non-lease current debt components. Manual review due to ambiguity from duplicate 'Bank loans' rows and possible overlap/inconsistency versus 'Liabilities Due within 1 Year' across years.,Balance Sheet::Bank loans; Balance Sheet::Mortgage Loans,4
COLOb.CO,STD,"Chose 'Other credit institutions' because it appears in the current-liabilities section and best matches short-term borrowings; excluded lease rows per rule. Manual review needed because the same exact row label also appears in the non-current liabilities section, creating ambiguity in row-label-only mapping.",Balance Sheet::Other credit institutions,1
COLOb.CO,TXP,"Selected 'Income tax' as the current tax liability/payable line. Manual review needed because the exact label 'Income tax' also appears again in the non-current liabilities section, so row-label-only mapping is ambiguous.",Balance Sheet::Income tax,3
COLUM.CO,STD,"Chose 'Debt to credit institutions' as the best available current debt line and excluded lease liabilities. Manual review because the same label appears in both current and non-current sections, creating some ambiguity across years.",Balance Sheet::Debt to credit institutions,3
DEMANT.CO,TXP,"'Income tax' appears in both asset and liability sections with the same exact label. The liability-side occurrence likely represents taxes payable/current tax liabilities, but the duplicated label prevents full confidence.",Balance Sheet::Income tax,1
DFDS.CO,STD,"Selected an exact, unique debt-current-portion line outside the shortlist because it explicitly excludes leases. Manual review needed because other current interest-bearing debt may also exist in separate rows, but duplicated 'Interest-bearing liabilities' labels are ambiguous and could cause double counting.",Balance Sheet::Current Portion of Long-Term Debt excluding Capitalized Leases,3
DNORD.CO,STD,"Chose current 'Borrowings' plus 'Current maturities' to capture short-term debt excluding lease liabilities. Manual review needed because 'Borrowings' may already include some current maturities in certain years, creating possible overlap.",Balance Sheet::Borrowings; Balance Sheet::Current maturities,2
ENSG.CO,STD,"Included current maturities and bank debt as the minimal financing-related current debt set, excluding lease liabilities. Manual review because duplicated 'Bank debts' labels in the statement create some ambiguity about exact scope and potential overlap.",Balance Sheet::Current part of long term interest-bearing debt; Balance Sheet::Bank debts,4
